# Parametric CAD Bench v2 + v3 — agentic pipeline (Qwen3.5, FreeCAD 1.1)

Run all cells with a GPU runtime (Runtime → Change runtime type → GPU; L4/A100 runs Qwen3.5-9B, T4 runs Qwen3.5-4B). Everything is saved to Google Drive under `MyDrive/cadbench-pipeline/`: if the session disconnects, reconnect and run all cells again — finished generations are reused. Graders and reference models are deleted from this copy before the model runs; scoring happens afterwards with the official verifiers.

In [ ]:
import subprocess, torch
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/cadbench-pipeline'
except Exception as e:  # e.g. a VS Code Colab kernel without Drive auth: results stay in /content and must be downloaded
    print('Drive not mounted:', e); OUT = '/content/cadbench-pipeline'
assert torch.cuda.is_available(), 'Select a GPU runtime first (Runtime > Change runtime type)'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'output ->', OUT)
subprocess.run(['mkdir', '-p', OUT + '/out'])

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.18.0', 'accelerate', 'pillow', 'uv'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
print('ok')

In [ ]:
import base64, pathlib
data = pathlib.Path('/content/data'); data.mkdir(exist_ok=True)
BLOBS = {"cadbench_pipeline.py": "...", "cadbench_pipeline_kaggle.py": "IiIiQmF0Y2hlZCBQYXJhbWV0cmljIENBRCBCZW5jaCB2Mit2MyBwaXBlbGluZSBmb3Igb25lIEdQVSBzZXNzaW9uIChLYWdnbGUgb3IgQ29sYWIpOiBRd2VuMy41ICg5QiBpbiBiZjE2IHdoZW4gYSBHUFUKaGFzID49IDIyIEdCLCBvdGhlcndpc2UgNEIgaW4gZnAxNjsgaXQgcmVhZHMgdGhlIGRyYXdpbmdzKSBhbmQgRnJlZUNBRCAxLjEuMCBmcm9tIGNvbmRhLWZvcmdlIChtaWNyb21hbWJhKSB3aXRoIHRoZQpwaW5uZWQgdmFsaWRhdG9yIGZvciB0aGUgcmVmZXJlbmNlLWZyZWUgaW50ZWdyaXR5IGdhdGVzLiBFdmVyeSBnZW5lcmF0ZWQgc2NyaXB0IGlzIGNhY2hlZCBpbiB0aGUgb3V0cHV0IGRpcmVjdG9yeSwgc28gYQpkaXNjb25uZWN0ZWQgc2Vzc2lvbiByZXN1bWVzIHdpdGhvdXQgcmVnZW5lcmF0aW5nOyBwcm9ncmVzcy5qc29uIHRoZXJlIHRyYWNrcyB0aGUgcnVuLiBHcmFkZXJzIGFuZCByZWZlcmVuY2UgbW9kZWxzIGFyZSBuZXZlciB1cGxvYWRlZDsgc2NvcmluZyBoYXBwZW5zIGFmdGVyd2FyZHMgd2l0aCB0aGUgb2ZmaWNpYWwKdmVyaWZpZXJzIChjYWRiZW5jaF9waXBlbGluZS5weSBzY29yZSkuCgpSb3VuZHMga2VlcCB0aGUgR1BVcyBidXN5OiAoMSkgc2luZ2xlIGFuc3dlciAoZ3JlZWR5LCB0aGUgemVyby1zaG90IGJhc2VsaW5lKSBhbmQgb25lIHNhbXBsZWQgYWx0ZXJuYXRpdmUgZm9yIGV2ZXJ5CnRhc2s7ICgyLi4pIGRpYWdub3Npcy1kcml2ZW4gcmVwYWlyIG9mIGV2ZXJ5IGZhaWxpbmcgYnJhbmNoOyB0aGVuIHNlbGVjdGlvbiBieSBnZW9tZXRyaWMgYWdyZWVtZW50IGFtb25nIHBhc3NpbmcKY2FuZGlkYXRlcyAoY2FkYmVuY2hfcGlwZWxpbmUuc29sdmUgbG9naWMpLgoKICBweXRob24gY2FkYmVuY2hfcGlwZWxpbmVfa2FnZ2xlLnB5IHBhY2thZ2UgICAgICAgICAgICAgICAgICAgICAobG9jYWwpIHRhc2tzLmpzb25sICsgZHJhd2luZ3MgLT4gdG1wL2thZ2dsZS1jYWRiZW5jaC9kYXRhCiAgcHl0aG9uIGNhZGJlbmNoX3BpcGVsaW5lX2thZ2dsZS5weSBydW4gLS1kYXRhIERJUiAtLW91dCBESVIgICAgKEthZ2dsZSkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQpmcm9tIGNvbmN1cnJlbnQuZnV0dXJlcyBpbXBvcnQgVGhyZWFkUG9vbEV4ZWN1dG9yCmltcG9ydCBqc29uCmltcG9ydCBvcwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKaW1wb3J0IHNodXRpbAppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgpIRVJFID0gUGF0aChfX2ZpbGVfXykucmVzb2x2ZSgpLnBhcmVudAoKCmRlZiBwYWNrYWdlKCkgLT4gTm9uZToKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoSEVSRSkpCiAgICBpbXBvcnQgY2FkYmVuY2hfcGlwZWxpbmUgYXMgY3AKICAgIGRhdGEgPSBIRVJFLnBhcmVudCAvICJ0bXAva2FnZ2xlLWNhZGJlbmNoL2RhdGEiCiAgICAoZGF0YSAvICJpbWFnZXMiKS5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICByb3dzID0gW10KICAgIGZvciB0IGluIGNwLnRhc2tzKFsidjIiLCAidjMiXSk6CiAgICAgICAgaW1ncyA9IFtdCiAgICAgICAgZm9yIGltZyBpbiB0WyJpbWFnZXMiXToKICAgICAgICAgICAgc2h1dGlsLmNvcHkoaW1nLCBkYXRhIC8gImltYWdlcyIgLyBQYXRoKGltZykubmFtZSk7IGltZ3MuYXBwZW5kKFBhdGgoaW1nKS5uYW1lKQogICAgICAgIHJvd3MuYXBwZW5kKHtrOiB0W2tdIGZvciBrIGluICgiaWQiLCAiYmVuY2giLCAidGFzayIsICJraW5kIiwgImZpbGVzIiwgImluc3RydWN0aW9uIil9IHwgeyJpbWFnZXMiOiBpbWdzfSkKICAgIChkYXRhIC8gInRhc2tzLmpzb25sIikud3JpdGVfdGV4dCgiIi5qb2luKGpzb24uZHVtcHMocikgKyAiXG4iIGZvciByIGluIHJvd3MpKQogICAgZm9yIGYgaW4gKCJjYWRiZW5jaF9waXBlbGluZS5weSIsICJjYWRiZW5jaF9waXBlbGluZV9rYWdnbGUucHkiKToKICAgICAgICBzaHV0aWwuY29weShIRVJFIC8gZiwgZGF0YSAvIGYpCiAgICAoZGF0YSAvICJkYXRhc2V0LW1ldGFkYXRhLmpzb24iKS53cml0ZV90ZXh0KGpzb24uZHVtcHMoZGljdCh0aXRsZT0iQ0FEIGJlbmNoIHBpcGVsaW5lIGlucHV0cyIsIGlkPSJheXVzaGRlYm5hdGgwMTIzL2NhZC1iZW5jaC1waXBlbGluZS1pbnB1dHMiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxpY2Vuc2VzPVt7Im5hbWUiOiAiQ0MwLTEuMCJ9XSkpKQogICAgcHJpbnQoanNvbi5kdW1wcyhkaWN0KHRhc2tzPWxlbihyb3dzKSwgaW1hZ2VzPXN1bShsZW4oclsiaW1hZ2VzIl0pIGZvciByIGluIHJvd3MpKSkpCgoKZGVmIHNldHVwX2ZyZWVjYWQocHJlZml4OiBQYXRoLCB3b3JrOiBQYXRoKSAtPiBOb25lOgogICAgaWYgKHByZWZpeCAvICJiaW4vcHl0aG9uIikuZXhpc3RzKCk6CiAgICAgICAgcmV0dXJuCiAgICBtbSA9IHdvcmsgLyAibWljcm9tYW1iYSIKICAgIGlmIG5vdCBtbS5leGlzdHMoKToKICAgICAgICBzdWJwcm9jZXNzLnJ1bihmImN1cmwgLUxzIGh0dHBzOi8vbWljcm8ubWFtYmEucG0vYXBpL21pY3JvbWFtYmEvbGludXgtNjQvbGF0ZXN0IHwgdGFyIC14dmogLUMge3dvcmt9IGJpbi9taWNyb21hbWJhIgogICAgICAgICAgICAgICAgICAgICAgIGYiICYmIG12IHt3b3JrfS9iaW4vbWljcm9tYW1iYSB7bW19Iiwgc2hlbGw9VHJ1ZSwgY2hlY2s9VHJ1ZSkKICAgIHN1YnByb2Nlc3MucnVuKFtzdHIobW0pLCAiY3JlYXRlIiwgIi15IiwgIi1xIiwgIi1wIiwgc3RyKHByZWZpeCksICItYyIsICJjb25kYS1mb3JnZSIsICJweXRob249My4xMiIsICJmcmVlY2FkPTEuMS4wIiwgIm51bXB5PTIuNC42IiwKICAgICAgICAgICAgICAgICAgICAib2NwPTcuOS4zLjEiLCAic2NpcHkiLCAicHlkYW50aWMiXSwgY2hlY2s9VHJ1ZSkKICAgIHNpdGUgPSBzdWJwcm9jZXNzLnJ1bihbc3RyKHByZWZpeCAvICJiaW4vcHl0aG9uIiksICItYyIsICJpbXBvcnQgc2l0ZTsgcHJpbnQoc2l0ZS5nZXRzaXRlcGFja2FnZXMoKVswXSkiXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICB0ZXh0PVRydWUsIGNoZWNrPVRydWUpLnN0ZG91dC5zdHJpcCgpCiAgICBQYXRoKHNpdGUsICJmcmVlY2FkLnB0aCIpLndyaXRlX3RleHQoc3RyKHByZWZpeCAvICJsaWIiKSArICJcbiIpCiAgICBzdWJwcm9jZXNzLnJ1bihbc3RyKHByZWZpeCAvICJiaW4vcHl0aG9uIiksICItbSIsICJwaXAiLCAiaW5zdGFsbCIsICItcSIsICJnbnVjbGV1cy1mcmVlY2FkLXZhbGlkYXRvcj09MC42LjAiXSwgY2hlY2s9VHJ1ZSkKICAgIHN1YnByb2Nlc3MucnVuKFtzdHIocHJlZml4IC8gImJpbi9weXRob24iKSwgIi1jIiwgImltcG9ydCBGcmVlQ0FELCBmcmVlY2FkX3ZhbGlkYXRvcjsgcHJpbnQoJ0ZyZWVDQUQnLCBGcmVlQ0FELlZlcnNpb24oKVs6M10pIl0sIGNoZWNrPVRydWUpCgoKY2xhc3MgR2VuZXJhdG9yOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsOiBzdHIsIG1heF9uZXc6IGludCwgYmF0Y2g6IGludCwgZGVhZGxpbmU6IGZsb2F0LCBjYWNoZTogUGF0aCk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IEF1dG9Nb2RlbEZvckltYWdlVGV4dFRvVGV4dCwgQXV0b1Byb2Nlc3NvcgogICAgICAgIHNlbGYudG9yY2ggPSB0b3JjaAogICAgICAgIGJpZyA9IG1heCh0b3JjaC5jdWRhLmdldF9kZXZpY2VfcHJvcGVydGllcyhpKS50b3RhbF9tZW1vcnkgZm9yIGkgaW4gcmFuZ2UodG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSkpID49IDIyZTkKICAgICAgICBiZjE2ID0gdG9yY2guY3VkYS5nZXRfZGV2aWNlX2NhcGFiaWxpdHkoMClbMF0gPj0gOAogICAgICAgIGlmIG1vZGVsID09ICJhdXRvIjogICMgb25lIFQ0IGNhbm5vdCBob2xkIHRoZSA5QiBtb2RlbCBpbiAxNi1iaXQKICAgICAgICAgICAgbW9kZWwgPSAiUXdlbi9Rd2VuMy41LTlCIiBpZiBiaWcgb3IgdG9yY2guY3VkYS5kZXZpY2VfY291bnQoKSA+IDEgZWxzZSAiUXdlbi9Rd2VuMy41LTRCIgogICAgICAgIHNlbGYubmFtZSwgZHR5cGUgPSBtb2RlbCwgKHRvcmNoLmJmbG9hdDE2IGlmIGJmMTYgZWxzZSB0b3JjaC5mbG9hdDE2KQogICAgICAgIHByaW50KCJNT0RFTCIsIG1vZGVsLCBkdHlwZSwgW3RvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKGkpIGZvciBpIGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpXSwgZmx1c2g9VHJ1ZSkKICAgICAgICBzZWxmLnByb2MgPSBBdXRvUHJvY2Vzc29yLmZyb21fcHJldHJhaW5lZChtb2RlbCkKICAgICAgICBzZWxmLnByb2MudG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJsZWZ0IgogICAgICAgIHNlbGYubW9kZWwgPSBBdXRvTW9kZWxGb3JJbWFnZVRleHRUb1RleHQuZnJvbV9wcmV0cmFpbmVkKG1vZGVsLCBkdHlwZT1kdHlwZSwgZGV2aWNlX21hcD0iYXV0byIpLmV2YWwoKQogICAgICAgIHNlbGYubWF4X25ldywgc2VsZi5iYXRjaCwgc2VsZi5kZWFkbGluZSA9IG1heF9uZXcsIGJhdGNoLCBkZWFkbGluZQogICAgICAgIHNlbGYuY2FjaGVfcGF0aCA9IGNhY2hlCiAgICAgICAgc2VsZi5jYWNoZSA9IHtqc29uLmxvYWRzKGwpWyJrZXkiXToganNvbi5sb2FkcyhsKVsidGV4dCJdIGZvciBsIGluIGNhY2hlLnJlYWRfdGV4dCgpLnNwbGl0bGluZXMoKX0gaWYgY2FjaGUuZXhpc3RzKCkgZWxzZSB7fQoKICAgIGRlZiBfX2NhbGxfXyhzZWxmLCByZXF1ZXN0czogbGlzdFtkaWN0XSkgLT4gbGlzdFtzdHJdOgogICAgICAgICIiInJlcXVlc3RzOiB7bWVzc2FnZXMsIGltYWdlcyAocGF0aHMpLCBzYW1wbGV9OyByZXR1cm5zIHRleHRzIGluIG9yZGVyIChiYXRjaGVkIGJ5IGxlbmd0aCkuIiIiCiAgICAgICAgZnJvbSBQSUwgaW1wb3J0IEltYWdlCiAgICAgICAgb3V0ID0gW3NlbGYuY2FjaGUuZ2V0KHJbImtleSJdKSBmb3IgciBpbiByZXF1ZXN0c10KICAgICAgICBvcmRlciA9IHNvcnRlZCgoaSBmb3IgaSBpbiByYW5nZShsZW4ocmVxdWVzdHMpKSBpZiBvdXRbaV0gaXMgTm9uZSksIGtleT1sYW1iZGEgaTogbGVuKGpzb24uZHVtcHMocmVxdWVzdHNbaV1bIm1lc3NhZ2VzIl0pKSkKICAgICAgICBmb3IgcyBpbiByYW5nZSgwLCBsZW4ob3JkZXIpLCBzZWxmLmJhdGNoKToKICAgICAgICAgICAgaWR4ID0gb3JkZXJbczpzICsgc2VsZi5iYXRjaF0KICAgICAgICAgICAgaWYgdGltZS50aW1lKCkgPiBzZWxmLmRlYWRsaW5lOiAgIyBsZWF2ZSB0aW1lIHRvIGZpbmFsaXplIGFuZCBzYXZlIGJlZm9yZSB0aGUgc2Vzc2lvbiBsaW1pdAogICAgICAgICAgICAgICAgZm9yIGkgaW4gaWR4OgogICAgICAgICAgICAgICAgICAgIG91dFtpXSA9ICIiCiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBmb3IgaSwgdCBpbiB6aXAoaWR4LCBzZWxmLl9nZW5lcmF0ZShbcmVxdWVzdHNbaV0gZm9yIGkgaW4gaWR4XSkpOgogICAgICAgICAgICAgICAgICAgIG91dFtpXSA9IHQKICAgICAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiAgIyBub3FhOiBCTEUwMDEgLSByZXRyeSBvbmUgYnkgb25lOyBhIHJlcXVlc3QgdGhhdCBzdGlsbCBmYWlscyBpcyBhbiBlbXB0eSBhbnN3ZXIKICAgICAgICAgICAgICAgIHByaW50KCJCQVRDSCBGQUlMRUQiLCB0eXBlKGUpLl9fbmFtZV9fLCBzdHIoZSlbOjMwMF0sIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICBmb3IgaSBpbiBpZHg6CiAgICAgICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgICAgICBvdXRbaV0gPSBzZWxmLl9nZW5lcmF0ZShbcmVxdWVzdHNbaV1dKVswXQogICAgICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZTI6ICAjIG5vcWE6IEJMRTAwMQogICAgICAgICAgICAgICAgICAgICAgICBwcmludCgiUkVRVUVTVCBGQUlMRUQiLCB0eXBlKGUyKS5fX25hbWVfXywgc3RyKGUyKVs6MzAwXSwgZmx1c2g9VHJ1ZSk7IG91dFtpXSA9ICIiCiAgICAgICAgICAgIHdpdGggc2VsZi5jYWNoZV9wYXRoLm9wZW4oImEiKSBhcyBoOgogICAgICAgICAgICAgICAgZm9yIGkgaW4gaWR4OgogICAgICAgICAgICAgICAgICAgIGlmIG91dFtpXToKICAgICAgICAgICAgICAgICAgICAgICAgc2VsZi5jYWNoZVtyZXF1ZXN0c1tpXVsia2V5Il1dID0gb3V0W2ldCiAgICAgICAgICAgICAgICAgICAgICAgIGgud3JpdGUoanNvbi5kdW1wcyhkaWN0KGtleT1yZXF1ZXN0c1tpXVsia2V5Il0sIHRleHQ9b3V0W2ldKSkgKyAiXG4iKQogICAgICAgICAgICBwcmludChmIkdFTkVSQVRFRCB7bWluKHMgKyBzZWxmLmJhdGNoLCBsZW4ob3JkZXIpKX0ve2xlbihvcmRlcil9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICAgICAgc2VsZi5wcm9ncmVzcyhkaWN0KGdlbmVyYXRlZD1taW4ocyArIHNlbGYuYmF0Y2gsIGxlbihvcmRlcikpLCBvZj1sZW4ob3JkZXIpKSkKICAgICAgICByZXR1cm4gb3V0CgogICAgZGVmIHByb2dyZXNzKHNlbGYsIGluZm86IGRpY3QpIC0+IE5vbmU6CiAgICAgICAgcCA9IHNlbGYuY2FjaGVfcGF0aC53aXRoX25hbWUoInByb2dyZXNzLmpzb24iKQogICAgICAgIHN0YXRlID0ganNvbi5sb2FkcyhwLnJlYWRfdGV4dCgpKSBpZiBwLmV4aXN0cygpIGVsc2Uge30KICAgICAgICBzdGF0ZS51cGRhdGUoaW5mbywgbW9kZWw9c2VsZi5uYW1lLCB0aW1lPXRpbWUuc3RyZnRpbWUoIiVZLSVtLSVkICVIOiVNOiVTIikpCiAgICAgICAgcC53cml0ZV90ZXh0KGpzb24uZHVtcHMoc3RhdGUpKQoKICAgIGRlZiBfZ2VuZXJhdGUoc2VsZiwgcmVxczogbGlzdFtkaWN0XSkgLT4gbGlzdFtzdHJdOgogICAgICAgIGZyb20gUElMIGltcG9ydCBJbWFnZQogICAgICAgIHRleHRzLCBpbWFnZXMgPSBbXSwgW10KICAgICAgICBpZiBUcnVlOgogICAgICAgICAgICBmb3IgciBpbiByZXFzOgogICAgICAgICAgICAgICAgbXNncyA9IFtkaWN0KG0pIGZvciBtIGluIHJbIm1lc3NhZ2VzIl1dCiAgICAgICAgICAgICAgICBpZiByWyJpbWFnZXMiXToKICAgICAgICAgICAgICAgICAgICBsYXN0ID0gbXNnc1stMV0KICAgICAgICAgICAgICAgICAgICBtc2dzWy0xXSA9IHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiBbeyJ0eXBlIjogImltYWdlIn0gZm9yIF8gaW4gclsiaW1hZ2VzIl1dICsgW3sidHlwZSI6ICJ0ZXh0IiwgInRleHQiOiBsYXN0WyJjb250ZW50Il19XX0KICAgICAgICAgICAgICAgICAgICBpbWFnZXMgKz0gW0ltYWdlLm9wZW4ocCkuY29udmVydCgiUkdCIikgZm9yIHAgaW4gclsiaW1hZ2VzIl1dCiAgICAgICAgICAgICAgICB0ZXh0cy5hcHBlbmQoc2VsZi5wcm9jLmFwcGx5X2NoYXRfdGVtcGxhdGUobXNncywgYWRkX2dlbmVyYXRpb25fcHJvbXB0PVRydWUsIHRva2VuaXplPUZhbHNlLCBlbmFibGVfdGhpbmtpbmc9RmFsc2UpKQogICAgICAgICAgICBlbmMgPSBzZWxmLnByb2ModGV4dD10ZXh0cywgaW1hZ2VzPWltYWdlcyBvciBOb25lLCByZXR1cm5fdGVuc29ycz0icHQiLCBwYWRkaW5nPVRydWUpLnRvKHNlbGYubW9kZWwuZGV2aWNlKQogICAgICAgICAgICBzYW1wbGUgPSBhbnkoclsic2FtcGxlIl0gZm9yIHIgaW4gcmVxcykKICAgICAgICAgICAgd2l0aCBzZWxmLnRvcmNoLmluZmVyZW5jZV9tb2RlKCk6CiAgICAgICAgICAgICAgICBnZW4gPSBzZWxmLm1vZGVsLmdlbmVyYXRlKCoqZW5jLCBtYXhfbmV3X3Rva2Vucz1zZWxmLm1heF9uZXcsIGRvX3NhbXBsZT1zYW1wbGUsIHRlbXBlcmF0dXJlPTAuOCBpZiBzYW1wbGUgZWxzZSBOb25lLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0b3BfcD0wLjk1IGlmIHNhbXBsZSBlbHNlIE5vbmUsIHBhZF90b2tlbl9pZD1zZWxmLnByb2MudG9rZW5pemVyLnBhZF90b2tlbl9pZCkKICAgICAgICAgICAgcmV0dXJuIHNlbGYucHJvYy5iYXRjaF9kZWNvZGUoZ2VuWzosIGVuY1siaW5wdXRfaWRzIl0uc2hhcGVbMV06XSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQoKCmRlZiBydW4oYSkgLT4gTm9uZToKICAgIHN5cy5wYXRoLmluc2VydCgwLCBzdHIoUGF0aChhLmRhdGEpKSkKICAgIGltcG9ydCBjYWRiZW5jaF9waXBlbGluZSBhcyBjcAogICAgd29yayA9IFBhdGgoYS53b3JrKTsgd29yay5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICBwcmVmaXggPSB3b3JrIC8gImZjIgogICAgc2V0dXBfZnJlZWNhZChwcmVmaXgsIHdvcmspCiAgICBjcC5GQ19QWVRIT04sIGNwLkZSRUVDQURfTElCID0gcHJlZml4IC8gImJpbi9weXRob24iLCBwcmVmaXggLyAibGliIgogICAgb3V0ID0gUGF0aChhLm91dCk7IG91dC5ta2RpcihwYXJlbnRzPVRydWUsIGV4aXN0X29rPVRydWUpCiAgICB0YXNrcyA9IFtqc29uLmxvYWRzKGwpIGZvciBsIGluIFBhdGgoYS5kYXRhLCAidGFza3MuanNvbmwiKS5yZWFkX3RleHQoKS5zcGxpdGxpbmVzKCldWzogYS5saW1pdCBvciBOb25lXQogICAgZm9yIHQgaW4gdGFza3M6CiAgICAgICAgdFsiaW1hZ2VzIl0gPSBbc3RyKFBhdGgoYS5kYXRhKSAvICJpbWFnZXMiIC8gcCkgZm9yIHAgaW4gdFsiaW1hZ2VzIl1dCiAgICAgICAgdFsidXNlciJdID0gdFsiaW5zdHJ1Y3Rpb24iXSArICgiXG5cblRoZSBkcmF3aW5nIGlzIGF0dGFjaGVkIGFuZCBpcyBzYXZlZCBuZXh0IHRvIHRoZSBzY3JpcHQgdW5kZXIgdGhlIHNhbWUgZmlsZSBuYW1lLiIgaWYgdFsiaW1hZ2VzIl0gZWxzZSAiIikKICAgIHRfc3RhcnQgPSB0aW1lLnRpbWUoKQogICAgZ2VuID0gR2VuZXJhdG9yKGEubW9kZWwsIGEubWF4X25ldywgYS5iYXRjaCwgZGVhZGxpbmU9dF9zdGFydCArIGEuaG91cnMgKiAzNjAwLCBjYWNoZT1vdXQgLyAiZ2VuZXJhdGlvbnMuanNvbmwiKQogICAgYnJhbmNoZXMgPSB7dFsiaWQiXTogW10gZm9yIHQgaW4gdGFza3N9ICAjIHBlciB0YXNrOiBsaXN0IG9mIGJyYW5jaGVzLCBlYWNoIGEgbGlzdCBvZiBub2RlcwogICAgcG9vbCA9IFRocmVhZFBvb2xFeGVjdXRvcig0KQoKICAgIGRlZiBldmFsdWF0ZSh0LCBraW5kLCB0ZXh0LCBicmFuY2gpOgogICAgICAgIHNjcmlwdCA9IGNwLmV4dHJhY3RfY29kZSh0ZXh0KQogICAgICAgIHJlcyA9IGNwLmV4ZWN1dGUodCwgc2NyaXB0KQogICAgICAgIG5vZGUgPSBkaWN0KGtpbmQ9a2luZCwgc2NyaXB0PXNjcmlwdCwgcmVzPXJlcywgb2s9Y3AucGFzc2VzKHJlcyksIGRpYWdub3Npcz1jcC5kaWFnbm9zaXMocmVzKSkKICAgICAgICBicmFuY2guYXBwZW5kKG5vZGUpCiAgICAgICAgcmV0dXJuIG5vZGUKICAgIHQwID0gdGltZS50aW1lKCkKICAgIHJlcXMsIG1ldGEgPSBbXSwgW10KICAgIGZvciB0IGluIHRhc2tzOgogICAgICAgIGZvciBrIGluIHJhbmdlKDEgKyBhLmFsdGVybmF0aXZlcyk6CiAgICAgICAgICAgIGhpbnQgPSAiIiBpZiBrID09IDAgZWxzZSAiXG5cbldyaXRlIGFuIGluZGVwZW5kZW50IGNvbnN0cnVjdGlvbjogY2hvb3NlIHRoZSBza2V0Y2ggcGxhbmVzLCBmZWF0dXJlIG9yZGVyIGFuZCBQYXJ0RGVzaWduIGZlYXR1cmVzIGFmcmVzaC4iCiAgICAgICAgICAgIHJlcXMuYXBwZW5kKGRpY3QobWVzc2FnZXM9W3sicm9sZSI6ICJzeXN0ZW0iLCAiY29udGVudCI6IGNwLlNZU1RFTX0sIHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiB0WyJ1c2VyIl0gKyBoaW50fV0sCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW1hZ2VzPXRbImltYWdlcyJdLCBzYW1wbGU9ayA+IDAsIGtleT1mInt0WydpZCddfXx7J3NpbmdsZScgaWYgayA9PSAwIGVsc2UgZidhbHRlcm5hdGl2ZXtrfSd9IikpCiAgICAgICAgICAgIG1ldGEuYXBwZW5kKCh0LCAic2luZ2xlIiBpZiBrID09IDAgZWxzZSBmImFsdGVybmF0aXZle2t9IikpCiAgICB0ZXh0cyA9IFtOb25lXSAqIGxlbihyZXFzKQogICAgZm9yIHNhbXBsZSBpbiAoRmFsc2UsIFRydWUpOiAgIyBncmVlZHkgc2luZ2xlIGFuc3dlcnMgYW5kIHNhbXBsZWQgYWx0ZXJuYXRpdmVzIG5ldmVyIHNoYXJlIGEgYmF0Y2gKICAgICAgICBzZWwgPSBbaSBmb3IgaSwgciBpbiBlbnVtZXJhdGUocmVxcykgaWYgclsic2FtcGxlIl0gPT0gc2FtcGxlXQogICAgICAgIGZvciBpLCB0ZXh0IGluIHppcChzZWwsIGdlbihbcmVxc1tpXSBmb3IgaSBpbiBzZWxdKSk6CiAgICAgICAgICAgIHRleHRzW2ldID0gdGV4dAogICAgam9icyA9IFtdCiAgICBmb3IgKHQsIGtpbmQpLCB0ZXh0IGluIHppcChtZXRhLCB0ZXh0cyk6ICAjIG9uZSBicmFuY2ggcGVyIGZpcnN0LXJvdW5kIGNhbmRpZGF0ZTsgdGhlIHNpbmdsZSBhbnN3ZXIgaXMgYnJhbmNoIDAKICAgICAgICBiID0gW10KICAgICAgICBicmFuY2hlc1t0WyJpZCJdXS5hcHBlbmQoYikKICAgICAgICBqb2JzLmFwcGVuZChwb29sLnN1Ym1pdChldmFsdWF0ZSwgdCwga2luZCwgdGV4dCwgYikpCiAgICBbai5yZXN1bHQoKSBmb3IgaiBpbiBqb2JzXQogICAgZ2VuLnByb2dyZXNzKGRpY3Qocm91bmQ9MCwgcGFzc2luZ19zaW5nbGVzPXN1bShicmFuY2hlc1t0WyJpZCJdXVswXVswXVsib2siXSBmb3IgdCBpbiB0YXNrcykpKQogICAgcHJpbnQoZiJST1VORCAwIGRvbmUgaW4ge3RpbWUudGltZSgpIC0gdDA6LjBmfXM7IHBhc3Npbmcgc2luZ2xlcyB7c3VtKGJyYW5jaGVzW3RbJ2lkJ11dWzBdWzBdWydvayddIGZvciB0IGluIHRhc2tzKX0ve2xlbih0YXNrcyl9IiwgZmx1c2g9VHJ1ZSkKICAgIGZvciByIGluIHJhbmdlKDEsIGEucmVwYWlycyArIDEpOgogICAgICAgIGlmIHRpbWUudGltZSgpID4gZ2VuLmRlYWRsaW5lOgogICAgICAgICAgICBwcmludCgiREVBRExJTkU6IHNraXBwaW5nIGZ1cnRoZXIgcmVwYWlyIHJvdW5kcyIsIGZsdXNoPVRydWUpOyBicmVhawogICAgICAgIHJlcXMsIG1ldGEgPSBbXSwgW10KICAgICAgICBmb3IgdCBpbiB0YXNrczoKICAgICAgICAgICAgZm9yIGIgaW4gYnJhbmNoZXNbdFsiaWQiXV06CiAgICAgICAgICAgICAgICBjdXIgPSBiWy0xXQogICAgICAgICAgICAgICAgaWYgbm90IGN1clsib2siXToKICAgICAgICAgICAgICAgICAgICByZXFzLmFwcGVuZChkaWN0KG1lc3NhZ2VzPVt7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBjcC5TWVNURU19LCB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogdFsidXNlciJdfSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB7InJvbGUiOiAiYXNzaXN0YW50IiwgImNvbnRlbnQiOiAiYGBgcHl0aG9uXG4iICsgY3VyWyJzY3JpcHQiXSArICJgYGAifSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB7InJvbGUiOiAidXNlciIsICJjb250ZW50IjogIkNoZWNrcyBvbiB0aGlzIHNjcmlwdCBmb3VuZDpcbi0gIiArICJcbi0gIi5qb2luKGN1clsiZGlhZ25vc2lzIl0pWzo0MDAwXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICArICJcblxuUmV0dXJuIHRoZSBjb3JyZWN0ZWQgY29tcGxldGUgc2NyaXB0LiJ9XSwgaW1hZ2VzPXRbImltYWdlcyJdLCBzYW1wbGU9RmFsc2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGtleT1mInt0WydpZCddfXxyZXBhaXJ7cn0tb2Yte2JbMF1bJ2tpbmQnXX0iKSkKICAgICAgICAgICAgICAgICAgICBtZXRhLmFwcGVuZCgodCwgYiwgZiJyZXBhaXJ7cn0tb2Yte2JbMF1bJ2tpbmQnXX0iKSkKICAgICAgICBpZiBub3QgcmVxczoKICAgICAgICAgICAgYnJlYWsKICAgICAgICB0ZXh0cyA9IGdlbihyZXFzKQogICAgICAgIFtmLnJlc3VsdCgpIGZvciBmIGluIFtwb29sLnN1Ym1pdChldmFsdWF0ZSwgdCwga2luZCwgdGV4dCwgYikgZm9yICh0LCBiLCBraW5kKSwgdGV4dCBpbiB6aXAobWV0YSwgdGV4dHMpXV0KICAgICAgICBwcmludChmIlJPVU5EIHtyfSBkb25lIGluIHt0aW1lLnRpbWUoKSAtIHQwOi4wZn1zOyByZXBhaXJlZCB7bGVuKHJlcXMpfSBicmFuY2hlcyIsIGZsdXNoPVRydWUpCiAgICAgICAgZ2VuLnByb2dyZXNzKGRpY3Qocm91bmQ9ciwgcmVwYWlyZWQ9bGVuKHJlcXMpKSkKICAgIHN1bW1hcnkgPSBbXQogICAgZm9yIHQgaW4gdGFza3M6CiAgICAgICAgbm9kZXMgPSBbbiBmb3IgYiBpbiBicmFuY2hlc1t0WyJpZCJdXSBmb3IgbiBpbiBiXQogICAgICAgIGZpcnN0ID0gYnJhbmNoZXNbdFsiaWQiXV1bMF1bMF0KICAgICAgICBjcC5zYXZlKG91dCwgInNpbmdsZSIsIHQsIGZpcnN0WyJzY3JpcHQiXSwgZmlyc3RbInJlcyJdKQogICAgICAgIG9rID0gW24gZm9yIG4gaW4gbm9kZXMgaWYgblsib2siXV0KICAgICAgICBwaWNrID0gZmlyc3QKICAgICAgICBpZiBvazoKICAgICAgICAgICAgbSA9IGNwLmlvdXMoW25bInJlcyJdWyJicmVwcyJdLmdldCh0WyJmaWxlcyJdWy0xXSkgZm9yIG4gaW4gb2tdKQogICAgICAgICAgICBzdXBwb3J0ID0gW3N1bSh4ID49IDAuOTkgZm9yIHggaW4gcm93KSBmb3Igcm93IGluIG1dCiAgICAgICAgICAgIHBpY2sgPSBva1ttYXgocmFuZ2UobGVuKG9rKSksIGtleT1sYW1iZGEgaTogKHN1cHBvcnRbaV0sIC1ub2Rlcy5pbmRleChva1tpXSkpKV0KICAgICAgICBjcC5zYXZlKG91dCwgInBpcGVsaW5lIiwgdCwgcGlja1sic2NyaXB0Il0sIHBpY2tbInJlcyJdKQogICAgICAgIHN1bW1hcnkuYXBwZW5kKGRpY3QoaWQ9dFsiaWQiXSwga2luZD10WyJraW5kIl0sIHNpbmdsZV9vaz1maXJzdFsib2siXSwgYW55X29rPWJvb2wob2spLCBwaWNrZWQ9cGlja1sia2luZCJdLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgbm9kZXM9W2RpY3Qoa2luZD1uWyJraW5kIl0sIG9rPW5bIm9rIl0sIGRpYWdub3Npcz1uWyJkaWFnbm9zaXMiXVs6Ml0pIGZvciBuIGluIG5vZGVzXSkpCiAgICAgICAgZCA9IG91dCAvICJ0cmFjZSIgLyB0WyJiZW5jaCJdIC8gdFsidGFzayJdOyBkLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICBmb3IgbiBpbiBub2RlczoKICAgICAgICAgICAgKGQgLyBmIntuWydraW5kJ119LnB5Iikud3JpdGVfdGV4dChuWyJzY3JpcHQiXSkKICAgIChvdXQgLyAic3VtbWFyeS5qc29ubCIpLndyaXRlX3RleHQoIiIuam9pbihqc29uLmR1bXBzKHMpICsgIlxuIiBmb3IgcyBpbiBzdW1tYXJ5KSkKICAgIGZpbmFsID0gZGljdCh0YXNrcz1sZW4odGFza3MpLCBzaW5nbGVfb2s9c3VtKHNbInNpbmdsZV9vayJdIGZvciBzIGluIHN1bW1hcnkpLCBwaXBlbGluZV9vaz1zdW0oc1siYW55X29rIl0gZm9yIHMgaW4gc3VtbWFyeSksCiAgICAgICAgICAgICAgICAgc2Vjb25kcz1yb3VuZCh0aW1lLnRpbWUoKSAtIHQwKSkKICAgIGdlbi5wcm9ncmVzcyhkaWN0KGRvbmU9VHJ1ZSwgKipmaW5hbCkpCiAgICBwcmludChqc29uLmR1bXBzKGZpbmFsKSwgZmx1c2g9VHJ1ZSkKCgpkZWYgbWFpbigpIC0+IE5vbmU6CiAgICBwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249X19kb2NfXywgZm9ybWF0dGVyX2NsYXNzPWFyZ3BhcnNlLlJhd0Rlc2NyaXB0aW9uSGVscEZvcm1hdHRlcikKICAgIHAuYWRkX2FyZ3VtZW50KCJjbWQiLCBjaG9pY2VzPSgicGFja2FnZSIsICJydW4iKSkKICAgIHAuYWRkX2FyZ3VtZW50KCItLWRhdGEiLCBkZWZhdWx0PSIuIik7IHAuYWRkX2FyZ3VtZW50KCItLW91dCIsIGRlZmF1bHQ9Ii9rYWdnbGUvd29ya2luZy9vdXQiKQogICAgcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJhdXRvIik7IHAuYWRkX2FyZ3VtZW50KCItLXdvcmsiLCBkZWZhdWx0PSIva2FnZ2xlL3dvcmtpbmciIGlmIFBhdGgoIi9rYWdnbGUiKS5leGlzdHMoKSBlbHNlICIvY29udGVudCIpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1tYXgtbmV3IiwgdHlwZT1pbnQsIGRlZmF1bHQ9NDA5Nik7IHAuYWRkX2FyZ3VtZW50KCItLWJhdGNoIiwgdHlwZT1pbnQsIGRlZmF1bHQ9NikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWFsdGVybmF0aXZlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTEpOyBwLmFkZF9hcmd1bWVudCgiLS1yZXBhaXJzIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MikKICAgIHAuYWRkX2FyZ3VtZW50KCItLWxpbWl0IiwgdHlwZT1pbnQpCiAgICBwLmFkZF9hcmd1bWVudCgiLS1ob3VycyIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MTAuMCwgaGVscD0ic3RvcCBnZW5lcmF0aW5nIGFmdGVyIHRoaXMgbWFueSBob3VycywgdGhlbiBzYXZlIikKICAgIGEgPSBwLnBhcnNlX2FyZ3MoKQogICAgcGFja2FnZSgpIGlmIGEuY21kID09ICJwYWNrYWdlIiBlbHNlIHJ1bihhKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK"}
for name, b in BLOBS.items():
    (data / name).write_bytes(base64.b64decode(b))
print(sorted(p.name for p in data.iterdir()))

In [ ]:
import json, shutil, subprocess, sys, pathlib
for v in ('v2', 'v3'):
    if not pathlib.Path(f'/content/harbor-{v}/cad-bench').exists():
        subprocess.run(['uvx', '--from', 'harbor', 'harbor', 'download', f'gnucleus-ai/cad-bench@{v}', '-o', f'/content/harbor-{v}'], check=True)
sys.path.insert(0, '/content/data')
import cadbench_pipeline as cp
cp.SUITES = {v: pathlib.Path(f'/content/harbor-{v}/cad-bench') for v in ('v2', 'v3')}
data = pathlib.Path('/content/data'); (data / 'images').mkdir(exist_ok=True)
rows = []
for t in cp.tasks(['v2', 'v3']):
    names = []
    for img in t['images']:
        shutil.copy(img, data / 'images' / pathlib.Path(img).name); names.append(pathlib.Path(img).name)
    rows.append({k: t[k] for k in ('id', 'bench', 'task', 'kind', 'files', 'instruction')} | {'images': names})
(data / 'tasks.jsonl').write_text(''.join(json.dumps(r) + '\n' for r in rows))
for suite in cp.SUITES.values():  # graders and references never stay next to the model
    for t in suite.iterdir():
        shutil.rmtree(t / 'tests', ignore_errors=True); shutil.rmtree(t / 'solution', ignore_errors=True)
print(len(rows), 'tasks;', sum(len(r['images']) for r in rows), 'drawings')

In [ ]:
import subprocess, sys
cmd = f"{sys.executable} -u /content/data/cadbench_pipeline_kaggle.py run --data /content/data --out {OUT}/out --work /content --hours 11 2>&1 | tee -a {OUT}/run.log"
subprocess.run(cmd, shell=True)
print(open(f'{OUT}/out/progress.json').read())